In [1]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 730, done.
remote: Counting objects: 100% (454/454), done.
remote: Compressing objects: 100% (280/280), done.
remote: Total 730 (delta 153), reused 226 (delta 71), pack-reused 276 (from 1)
Receiving objects: 100% (730/730), 4.43 MiB | 17.31 MiB/s, done.
Resolving deltas: 100% (227/227), done.
Mounted at /content/drive


In [2]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.8 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [4]:
from utils.datasets     import NF_IDS_Dataset
from utils.models       import EdgeGRU_Baseline_Entropy
from utils.metrics      import calculate_metrics_gnn
from utils.training     import train_epoch, evaluate, find_optimal_threshold, set_seed, run_multiple_seeds
from utils.experiment   import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots


# Auxiliary

In [5]:
ROOT_PATH = "./dataset_processed_entropy"

In [6]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [7]:
ROOT_DIR = "./results_earlystopping_entropy"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")

# Main

## Seeds

In [8]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Not used in EdgeGRU_Baseline_Entropy (node features replaced by node_stats)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [9]:
model_config = {
    "model_name": None,
    "type": "EdgeGRU_Baseline_Entropy",
    "model_params": {
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
EXPERIMENT_NAME="EdgeGRU_BiasOn_entropy"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [ ]:
run_multiple_seeds(
    model_class=EdgeGRU_Baseline_Entropy,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: EdgeGRU_BiasOn_entropy
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=EdgeGRU_BiasOn_entropy_seed42_20260416_130318

EdgeGRU_BiasOn_entropy_seed42
   Ep 1 | Loss: 0.2083 | Val Loss: 0.2337 | Val AUC-PR: 0.1139 (*)
   Ep 4 | Loss: 0.1875 | Val Loss: 0.2299 | Val AUC-PR: 0.1349 (*)
   Ep 6 | Loss: 0.1610 | Val Loss: 0.2121 | Val AUC-PR: 0.1534 (*)
   Ep 7 | Loss: 0.1614 | Val Loss: 0.2087 | Val AUC-PR: 0.1817 (*)
   Ep 9 | Loss: 0.1722 | Val Loss: 0.2043 | Val AUC-PR: 0.1923 (*)
   Ep 10 | Loss: 0.1525 | Val Loss: 0.2144 | Val AUC-PR: 0.1929 (*)
   Ep 11 | Loss: 0.1641 | Val Loss: 0.2094 | Val AUC-PR: 0.2047 (*)
   Ep 12 | Loss: 0.1685 | Val Loss: 0.1961 | Val AUC-PR: 0.2135 (*)
   Ep 13 | Loss: 0.1486 | Val Loss: 0.1954 | Val AUC-PR: 0.2252 (*)
   Ep 17 | Loss: 0.1444 | Val Loss: 0.1804 | Val AUC-PR: 0.2947 (*)
   Ep 18 | Loss: 0.1301 | Val Loss: 0.1783 | Val AUC-PR: 0.3206 (*)
   Ep 20 

## Save threshold

In [11]:
from utils.evaluation import compute_and_save_thresholds

EXPERIMENT_NAME="EdgeGRU_BiasOn_entropy"

compute_and_save_thresholds(
    model_class=EdgeGRU_Baseline_Entropy,
    model_config=model_config,
    val_loader=val_loader,
    experiment_name=EXPERIMENT_NAME,
    device=DEVICE,
    results_dir=ROOT_DIR
)

  seed 42: threshold = 0.5459536314010620
  seed 123: threshold = 0.3569243848323822
  seed 777: threshold = 0.1784505546092987
  seed 2024: threshold = 0.6520128250122070
  seed 99: threshold = 0.4472101032733917

Saved: ./results_earlystopping_entropy/logs/EdgeGRU_BiasOn_entropy/thresholds_EdgeGRU_BiasOn_entropy.npz
